# 🎬 AutoVSF Workstation Platform v2.0 Launcher
Run the cells below to mount Google Drive, bootstrap the Ubuntu Workstation environment, and launch the AutoVSF Desktop App.

In [ ]:
# Step 1: Connect to Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
WORKDIR = '/content/drive/MyDrive/AutoVSF'
os.makedirs(WORKDIR, exist_ok=True)
%cd $WORKDIR

if not os.path.exists('autovsf-colab-gui'):
    !git clone https://github.com/lionc2240/autovsf-colab-gui.git
%cd autovsf-colab-gui
!git pull

In [ ]:
# Step 2: Bootstrap Workstation Environment & Dependencies
!chmod +x bootstrap/setup_environment.sh
!./bootstrap/setup_environment.sh

In [ ]:
# Step 3: Launch Desktop GUI & noVNC Workstation Proxy
import subprocess, time
from google.colab.output import eval_js

# Start Virtual Framebuffer Xvfb
!pkill Xvfb || true
subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1280x800x24'])
os.environ['DISPLAY'] = ':1'
time.sleep(2)

# Start XFCE Desktop Window Manager
subprocess.Popen(['xfce4-session'], env=os.environ)
time.sleep(2)

# Start x11vnc & noVNC Web Server
subprocess.Popen(['x11vnc', '-display', ':1', '-forever', '-shared', '-nopw', '-rfbport', '5900'])
subprocess.Popen(['websockify', '--web=/usr/share/novnc/', '6080', 'localhost:5900'])
time.sleep(2)

# Launch AutoVSF Desktop GUI App
subprocess.Popen(['python3', '-m', 'autovsf_gui.app'], env=os.environ)

novnc_url = eval_js('google.colab.kernel.proxyPort(6080)')
print('='*70)
print(f'👉 ACCESS UBUNTU WORKSTATION DESKTOP: {novnc_url}/vnc.html')
print('='*70)